# Validación visual del preprocesado — una estrella, todos los pasos

Objetivo: tomar **una** curva de luz TESS y ver cada paso del pipeline de
preprocesado funcionando:

1. curva cruda (con gaps marcados)
2. sigma-clipping cerca de gaps/bordes — `msv.cleaning.sigma_clip_gap_edges`
3. curva final vs cruda — `msv.cleaning.clean_lightcurve`
4. periodogramas LS + ACF con detección de picos — `msv.periodograms` + `msv.peaks`
5. phase-fold de **todos** los peaks detectados

El notebook solo orquesta: toda la lógica vive en `src/msv`. Si algo se ve mal
aquí, el fix va en el paquete, no en el notebook. Env: **CNN_TESS** (no
necesita TF). Para la revisión de punta a punta con CNN+BRF ver
`4_Visual_Review_Pipeline.ipynb`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from msv import config, viz
from msv.cleaning import sigma_clip_gap_edges, clean_lightcurve

# ---- Estrella a revisar ------------------------------------------------------
# Default: primer par de config.FP_PAIRS (falsos positivos conocidos del corte
# Number_DST: ideales para auditar la limpieza de telemetría). OJO:
# los FP_PAIRS viven en el parquet de OGLE, no en el de masivas.
# Cambiar a mano, o usar viz.random_pairs(1)[0] para una estrella al azar.
TIC, SECTOR = config.FP_PAIRS[0]
LC_PARQUET = config.LC_PARQUET_OGLE             # o config.LC_PARQUET_MASSIVE

lc = viz.load_lc(TIC, SECTOR, LC_PARQUET)
t_raw = lc["Time"].to_numpy()
f_raw = lc["flux"].to_numpy()
e_raw = lc["flux_err"].to_numpy()
print(f"TIC {TIC} — sector {SECTOR}: {len(lc)} puntos crudos")

## Paso 0 — curva cruda

Gaps > 1 día sombreados en gris: son las zonas donde la telemetría deja
outliers, y donde va a actuar el sigma-clip del paso 1.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(t_raw, f_raw, ".", color="k", ms=2)
for g in np.where(np.diff(t_raw) > 1.0)[0]:
    ax.axvspan(t_raw[g], t_raw[g + 1], color="0.9", zorder=0)
ax.set_xlabel("Time [BTJD]")
ax.set_ylabel("PDCSAP flux")
ax.set_title(f"TIC {TIC} s{SECTOR} — curva cruda (gaps > 1 d sombreados)")
ax.grid(alpha=0.3)
plt.show()

## Paso 1 — sigma-clipping cerca de gaps y bordes (`sigma_clip_gap_edges`)

La mediana/σ de referencia se calculan con la curva completa **excluyendo**
las zonas vecinas a gaps/bordes; el rechazo a 3σ se aplica **solo** a los
puntos dentro de esas zonas — el resto de la curva queda intacto. Abajo,
zoom a cada zona donde el clipping puede actuar.

In [ ]:
t2, f2, keep2 = sigma_clip_gap_edges(t1[keep1], f1[keep1], return_mask=True)
dropped = ~keep2
med, std = np.median(f2[keep2]), np.std(f2[keep2])
print(f"sigma-clip bordes: {dropped.sum()} puntos clippeados de {len(t2)}")

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(t2[keep2], f2[keep2], ".", color="k", ms=2,
        label=f"conservados ({keep2.sum()})")
if dropped.any():
    ax.plot(t2[dropped], f2[dropped], "x", color="tab:red", ms=7, mew=1.5,
            label=f"clippeados ({dropped.sum()})")
ax.axhline(med, color="tab:blue", lw=1, label="mediana ± 3σ (aprox. de la referencia)")
for s in (-3, 3):
    ax.axhline(med + s * std, color="tab:blue", ls=":", lw=1)
gaps = np.where(np.diff(t2) > 1.0)[0]
for g in gaps:
    ax.axvspan(t2[g], t2[g + 1], color="0.9", zorder=0)
ax.set_xlabel("Time [BTJD]")
ax.set_ylabel("flux")
ax.set_title(f"TIC {TIC} s{SECTOR} — paso 2: sigma-clip en gaps/bordes")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.show()

# Zoom ±1 d alrededor de cada borde y gap (donde el clipping puede actuar)
edges = [t2[0], t2[-1]] + [t2[g] for g in gaps] + [t2[g + 1] for g in gaps]
edges = sorted(set(np.round(edges, 3)))[:8]
ncols = min(4, len(edges))
nrows = int(np.ceil(len(edges) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 2.7 * nrows),
                         squeeze=False)
for ax, t_edge in zip(axes.ravel(), edges):
    w = (t2 > t_edge - 1.0) & (t2 < t_edge + 1.0)
    ax.plot(t2[w & keep2], f2[w & keep2], ".", color="k", ms=3)
    ax.plot(t2[w & dropped], f2[w & dropped], "x", color="tab:red", ms=7, mew=1.5)
    ax.set_title(f"±1 d de t={t_edge:.2f}", fontsize=9)
    ax.grid(alpha=0.3)
for ax in axes.ravel()[len(edges):]:
    ax.axis("off")
fig.suptitle("Zoom: zonas vecinas a gaps/bordes", fontsize=11)
plt.tight_layout()
plt.show()

## Paso 2 — curva final vs cruda (`clean_lightcurve`)

`clean_lightcurve` es el entry point de producción: aplica los pasos 1 y 2 en
orden. Lo corrido arriba a mano debe coincidir con esto.

In [ ]:
tc, fc, ec = clean_lightcurve(t_raw, f_raw, e_raw)

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(t0, f0, ".", color="0.8", ms=2, label=f"cruda ({len(t0)})")
ax.plot(tc, fc, ".", color="k", ms=2, label=f"limpia ({len(tc)})")
ax.set_xlabel("Time [BTJD]")
ax.set_ylabel("PDCSAP flux")
ax.set_title(f"TIC {TIC} s{SECTOR} — paso 3: antes (gris) vs después (negro)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.show()

n_manual = int(keep2.sum())
assert len(tc) == n_manual, (len(tc), n_manual)
print(f"OK: pipeline completo = pasos 1+2 a mano ({len(tc)} puntos)")

## Paso 3 — periodogramas LS + ACF con detección de picos

Se computan **en vivo** sobre la curva limpia (mismo schema que los parquets
precalculados). Picos sobre el power crudo, sin suavizado: ventana mínima
`MIN_PEAK_SEP_DAYS = 0.5 d` y prominencia `ACF_PROMINENCE_FRAC = 5%` del
máximo (validado: 76% de recovery del período OGLE, 0 espurios en los FP).

In [ ]:
df_ls, df_acf = viz.compute_periodograms(tc, fc, ec)
df_ls, df_acf, top_ls, top_acf = viz.plot_periodograms_pair(
    TIC, SECTOR, dfs=(df_ls, df_acf))
plt.show()

print("Peaks LS:")
print(top_ls.to_string(index=False))
print()
print("Peaks ACF:")
print(top_acf.to_string(index=False))

## Paso 4 — phase-fold de TODOS los peaks

Dos ciclos por peak para ver la continuidad en fase 1. Un período real se ve
como estructura coherente; un armónico o alias se ve doblado o difuso.

In [ ]:
peaks = pd.concat([top_ls.assign(source="LS"), top_acf.assign(source="ACF")],
                  ignore_index=True).sort_values("prominence", ascending=False)

if peaks.empty:
    print("Sin peaks sobre el FAP: nada que plegar.")
else:
    ncols = 3
    nrows = int(np.ceil(len(peaks) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.2 * nrows),
                             squeeze=False)
    for ax, (_, r) in zip(axes.ravel(), peaks.iterrows()):
        viz.plot_phase_fold(
            tc, fc, r["per"], ax=ax,
            title=f"{r['source']}  P={r['per']:.4f} d  prom={r['prominence']:.2f}")
    for ax in axes.ravel()[len(peaks):]:
        ax.axis("off")
    fig.suptitle(f"TIC {TIC} s{SECTOR} — {len(peaks)} peaks phase-folded",
                 fontsize=12, y=1.001)
    plt.tight_layout()
    plt.show()

## Siguiente paso

Con el preprocesado validado, la revisión de punta a punta (hist2d + CNN+BRF
+ gate) para esta misma estrella es:

```python
viz.show_sample(TIC, SECTOR, lc_parquet=LC_PARQUET,
                brf_csv=config.RESULTS_DIR / 'brf_mc_peaks_final.csv')
```

o correr `4_Visual_Review_Pipeline.ipynb` sobre samples aleatorios.